# Preprocessing Speech (Chirac/Mitterrand)

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
import sys
import os

sys.path.append(os.path.abspath("../.."))
from utils import *

## Load text

alllabs: 
- 1: Chirac
- -1: Mitterrand

In [6]:
FILE_NAME = "../../../data/corpus.tache1.learn.utf8"

alltxts, alllabs = load_pres(FILE_NAME)

print(len(alltxts))
print(alltxts[10])
print(alllabs[10])

57413
 A Brazzaville, que l'Afrique de demain se dessine.

0


## Remove Numbers


In [7]:
processed_txts = remove_numbers(alltxts)

In [8]:
uninformative_words = find_uninformative_words(processed_txts, alllabs, threshold=0.95)
len(uninformative_words)

Filtering 449 uninformative/heavy words...


449

## Vectorize with TfidfVectorizer

**TF-IDF**: words can also be weighted by importance.  
Corpus: $C = \{\mathbf d_{1}, \ldots, \mathbf d_{|C|}\}$, vocabulary: $V = \{\mathbf w_{1}, \ldots, \mathbf w_{|V|}\}$:

- $\mathbf{d}_{ik}^{(tf)}$ term frequency for word $w_k$ in document $d_i$, s.t. $\sum_{k=1}^{|V|} d_{ik}^{(tf)} = 1$  
- $\mathrm{df}_{k}$ document frequency: $\mathrm{df}_{k} = \frac{|\{\mathbf d : w_{k} \in \mathbf d\}|}{|C|}$

TF-IDF for word $w_k$ in document $d_i$:

$$
d_{ik}^{(tfidf)} = d_{ik}^{(tf)} \, \log \frac{1}{\mathrm{df}_{k}}
$$

**Main parameters:**
- **use_idf:** boolean, default=True.  
- **smooth_idf:** Smooth idf weights, default=True. Adds one to document frequencies, as if an extra document was seen containing every term in the collection exactly once. Prevents division by zero.  
- **sublinear_tf:** boolean, default=False. Apply sublinear tf scaling, i.e., replace $d_{ik}^{(tf)}$ with $1 + \log(d_{ik}^{(tf)})$.

In [9]:
# Get French stopwords as a list
french_stopwords = stopwords.words('french') + uninformative_words
# Remove their accents
french_stopwords_no_accents = [remove_accents(word) for word in french_stopwords]
#french_stopwords_stemmed = stemming_french(french_stopwords_no_accents)

# ATTENTION: one sentence = one doc
vectorizer = TfidfVectorizer(
    lowercase=True,           # handles capitalization
    stop_words=french_stopwords_no_accents,     # removes stop words (Pass the stop words list)
    max_df=0.80,             # ignore terms in >95% of docs
    min_df=15,                # ignore terms in <2 docs
    ngram_range=(1, 2),       # unigrams + bigrams,
    strip_accents='unicode'  # handles accents
)

## Stemming (Optional)
Might hurt performance, test if needed

In [6]:
#processed_txts = stemming_french(alltxts)

In [10]:
X = vectorizer.fit_transform(processed_txts)

**n_features is vocaburary**: unique words across all documents

In [11]:
# X is a sparse matrix
print("Shape of X:", X.shape)  # (n_documents, n_features)
feature_names = vectorizer.get_feature_names_out()
print(feature_names[:10])

Shape of X: (57413, 5265)
['abandon' 'abandonner' 'aborde' 'aborder' 'aboutir' 'aboutissement'
 'abri' 'absence' 'absolue' 'absolument']


## Train / test split

In [12]:
from sklearn.model_selection import train_test_split

rs=10
[X_train, X_test, y_train, y_test]  = train_test_split(X, alllabs, test_size=0.2, random_state=rs, shuffle=True)


print(X_train.shape)
print(X_test.shape)
print(len(y_train))


(45930, 5265)
(11483, 5265)
45930


## Try on three models
- Naïve bayes
- Logistic Regression
- SVM

For now just fit each model below with default parameters

In [13]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score


#Naïve Bayes
nb_clf = MultinomialNB()
nb_clf.fit(X_train, y_train)


#Logistic Regression
t = 1e-8
C=100.0
lr_clf = LogisticRegression(random_state=0, solver='liblinear',max_iter=100, tol=t, C=C)
lr_clf.fit(X_train, y_train)

#Linear SVM
svm_clf = LinearSVC(random_state=0)
svm_clf.fit(X_train, y_train)

pred_nbt = nb_clf.predict(X_train)
pred_lrt = lr_clf.predict(X_train)
pred_svmt = svm_clf.predict(X_train)

pred_nb = nb_clf.predict(X_test)
pred_lr = lr_clf.predict(X_test)
pred_svm = svm_clf.predict(X_test)


evaluate_model("Naïve Bayes", nb_clf, X_test, y_test)
evaluate_model("Logistic Regression", lr_clf, X_test, y_test)
evaluate_model("Linear SVM", svm_clf, X_test, y_test)

--- Naïve Bayes ---
F1-Score: 0.1364
ROC-AUC:  0.8325
Avg Prec: 0.5134

--- Logistic Regression ---
F1-Score: 0.4515
ROC-AUC:  0.8190
Avg Prec: 0.4980

--- Linear SVM ---
F1-Score: 0.4510
ROC-AUC:  0.8307
Avg Prec: 0.5270

